# County Rainfall and Planting Advisory
**Author:** Asman Makero Musa · PLP Data Analytics Cohort 7

In [1]:
counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70]
]
machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ", "9", "12", "", "45", "182", "96"]
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

## Task 1: Clean Machakos Data with ValueError Handling

In [2]:
def clean_machakos(raw_data, month_names):
    """Clean text rainfall data, imputing invalid entries with rounded valid mean."""
    valid = []
    for v in raw_data:
        try:
            valid.append(int(v.strip()))
        except ValueError:
            pass
    avg_val = round(sum(valid) / len(valid))
    cleaned, imputed = [], []
    for i, v in enumerate(raw_data):
        try:
            cleaned.append(int(v.strip()))
        except ValueError:
            cleaned.append(avg_val)
            imputed.append(month_names[i])
    return cleaned, avg_val, imputed

machakos_clean, machakos_avg, imputed_months = clean_machakos(machakos_raw, months)
print(f"Imputed Value: {machakos_avg} mm | Imputed Months: {', '.join(imputed_months)}")
print(f"Cleaned Machakos: {machakos_clean}")

Imputed Value: 68 mm | Imputed Months: Apr, Sep
Cleaned Machakos: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


## Task 2: 5x12 NumPy Array

In [3]:
import numpy as np
rainfall = np.array(clean_rows + [machakos_clean])
print(f"Shape: {rainfall.shape} | Dtype: {rainfall.dtype}")

Shape: (5, 12) | Dtype: int64


## Task 3: Annual Totals and Monthly Averages

In [4]:
annual_totals = rainfall.sum(axis=1)
monthly_averages = rainfall.mean(axis=0)
print("Annual Totals (mm):", dict(zip(counties, annual_totals)))
print("Monthly Averages (mm):", dict(zip(months, monthly_averages.round(1))))

Annual Totals (mm): {'Nairobi': np.int64(1045), 'Nakuru': np.int64(916), 'Kisumu': np.int64(1371), 'Mombasa': np.int64(1075), 'Machakos': np.int64(811)}
Monthly Averages (mm): {'Jan': np.float64(47.0), 'Feb': np.float64(47.6), 'Mar': np.float64(91.0), 'Apr': np.float64(164.4), 'May': np.float64(171.2), 'Jun': np.float64(67.6), 'Jul': np.float64(56.6), 'Aug': np.float64(58.2), 'Sep': np.float64(64.2), 'Oct': np.float64(67.8), 'Nov': np.float64(127.4), 'Dec': np.float64(80.6)}


## Task 4: Wettest Month per County

In [5]:
wettest_idx = rainfall.argmax(axis=1)
wettest_months = [months[i] for i in wettest_idx]
print("Wettest Month per County:", dict(zip(counties, wettest_months)))

Wettest Month per County: {'Nairobi': 'Apr', 'Nakuru': 'Apr', 'Kisumu': 'Apr', 'Mombasa': 'May', 'Machakos': 'Nov'}


## Task 5: Dry-Month Counts (< 50 mm)

In [6]:
dry_mask = rainfall < 50
dry_counts = dry_mask.sum(axis=1)
print("Dry Months (<50mm) per County:", dict(zip(counties, dry_counts)))

Dry Months (<50mm) per County: {'Nairobi': np.int64(4), 'Nakuru': np.int64(3), 'Kisumu': np.int64(0), 'Mombasa': np.int64(2), 'Machakos': np.int64(6)}


## Task 6: Consecutive Two-Month Planting Windows

In [7]:
consec = rainfall[:, :-1] + rainfall[:, 1:]
best_idx = consec.argmax(axis=1)
planting_windows = [f"{months[i]}-{months[i+1]}" for i in best_idx]
print("Planting Windows:", dict(zip(counties, planting_windows)))

Planting Windows: {'Nairobi': 'Apr-May', 'Nakuru': 'Apr-May', 'Kisumu': 'Apr-May', 'Mombasa': 'Apr-May', 'Machakos': 'Nov-Dec'}


## Task 7: Export to CSV

In [8]:
import csv
rows = []
for c, t, w, d, win in zip(counties, annual_totals, wettest_months, dry_counts, planting_windows):
    rows.append({"county": c, "annual_total_mm": int(t), "wettest_month": w, "dry_months": int(d), "planting_window": win})

fieldnames = ["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"]
with open("rainfall_summary.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)
print("Successfully exported rainfall_summary.csv with 5 rows.")

Successfully exported rainfall_summary.csv with 5 rows.


# Planting Advisory

1. **Central & Western (Nairobi, Nakuru, Kisumu)**: Target planting in **Apr-May** when two-month rainfall peaks between 265 mm and 420 mm, with Kisumu recording the highest annual total at 1,371 mm and 0 dry months.
2. **Coast (Mombasa)**: Plant in **Apr-May** (390 mm combined), taking advantage of May as the wettest month (240 mm) to establish crops before the dry season.
3. **Eastern Semi-Arid (Machakos)**: Schedule planting strictly during **Nov-Dec** (278 mm combined), driven by the November peak (182 mm), as the county experiences 6 dry months below 50 mm.

# Data Quality Notes

- **Imputation**: Machakos data had non-numeric entries in April ('N/A') and September (''), imputed with the 10-month valid station mean (68 mm).
- **Seasonal Caveat**: Uniform mean imputation in April may underestimate peak long-rains moisture for Machakos.
- **Recommendation**: Verify station readings against multi-year Kenya Meteorological Department data before releasing farmer credit.